<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-weight:bold; color:#5A4636; text-align:center;">
AI Lab Recruitment Task 2<br>
<span style="font-size:22px;">Tabular Binary Classification — From Scratch</span>
</h1>

<p style="text-align:center; font-size:16px; color:#6F5C4F;">
Decision Tree Learning (CART) • Logistic Regression • Support Vector Machine
</p>

<p style="text-align:center; font-size:15px; color:#6F5C4F; line-height:1.7;">
Nama: <b>Kurt Mikhael Purba</b><br>
</p>


# Daftar Isi
1. [Pendahuluan](#1)
2. [Problem Statement & Ketentuan](#2)
3. [Dataset Overview](#3)
4. [Initialization](#4)
5. [Exploratory Data Analysis](#5)
6. [Preprocessing & Baseline Validation](#6)
7. [Implementasi From Scratch](#7)
8. [Pembanding Scikit-learn](#8)
9. [Evaluasi Baseline](#9)
10. [Robust Model Development — No Leakage](#10)
11. [Final Training & Kaggle Submission](#11)
12. [Kesimpulan](#12)
13. [Optional Post-Submission Audit](#13)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pendahuluan <a name="1"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Notebook ini menyelesaikan kasus **tabular binary classification** pada data kelayakan/risiko pinjaman. Target yang diprediksi adalah `loan_status`, dengan dua kelas:

- `0`: kelas negatif
- `1`: kelas positif

Tiga algoritma dibangun dari nol (**from scratch**) menggunakan operasi matematis NumPy:

1. **Decision Tree Learning — CART**, dengan pemilihan split berdasarkan Gini impurity.
2. **Logistic Regression**, dengan sigmoid, binary cross-entropy, regularisasi L2, dan optimisasi mini-batch Adam.
3. **Linear Support Vector Machine**, dengan hinge loss dan mini-batch subgradient descent.

Setelah itu, hasil ketiganya dibandingkan dengan implementasi sejenis dari scikit-learn menggunakan split validasi yang sama.


Versi pengembangan ini menambahkan **repeated stratified cross-validation**, **exact threshold tuning berbasis out-of-fold (OOF)**, pencarian split CART tanpa subsampling threshold, dan **nonlinear feature expansion yang target-independent** untuk Logistic Regression/SVM. Seluruh keputusan final tetap dibuat hanya dari `train.csv`; `test.csv` tidak pernah digunakan untuk fit, tuning, feature selection, atau threshold selection.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Problem Statement & Ketentuan <a name="2"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Problem Statement

Diberikan data karakteristik individu, profil kredit, dan informasi pinjaman. Tujuannya adalah memprediksi `loan_status` pada setiap baris `test.csv`.

## Ketentuan Implementasi

- Implementasi model from scratch hanya menggunakan library komputasi matematis, yaitu **NumPy**.
- Pandas digunakan untuk membaca dan mengelola tabel.
- Matplotlib digunakan untuk visualisasi.
- Scikit-learn hanya digunakan untuk:
  - metrik evaluasi;
  - implementasi algoritma pembanding;
  - bukan sebagai komponen internal model from scratch.
- `person_id` tidak digunakan sebagai fitur karena merupakan identifier, bukan karakteristik prediktif yang semestinya dipelajari.
- File submission harus memiliki kolom `person_id` dan `loan_status` dengan urutan identik terhadap `test.csv`.


## Prinsip Anti-Leakage pada Versi Robust

- `person_id` tetap dilarang sebagai fitur walaupun secara kebetulan dapat berkorelasi dengan target pada split tertentu.
- Statistik preprocessing, kategori, dan quantile cut-point hanya dipelajari dari training fold masing-masing.
- Threshold klasifikasi dipilih dari **OOF prediction train**, bukan dari test atau leaderboard.
- Label test tidak digunakan untuk pemilihan model/hyperparameter.
- Repeated CV hanya dipakai untuk tuning/evaluasi; submission tetap berasal dari **satu model manual**, bukan ensemble.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Dataset Overview <a name="3"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Struktur Direktori Kaggle

```text
/kaggle/input/competitions/ai-lab-recruitment-task-2/
├── train.csv
├── test.csv
└── sample_submission.csv
```

Notebook menyediakan fallback lokal sehingga tetap dapat diuji di luar Kaggle.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Initialization <a name="4"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">


## Setup, Seed, Path, dan Import


In [ ]:
from pathlib import Path
import sys

_project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(_project_root / "src"))
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

KAGGLE_DIR = Path("/kaggle/input/competitions/ai-lab-recruitment-task-2")
LOCAL_DIR = Path("/mnt/data/ai_lab_task2")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR   :", DATA_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)


## Load Dataset


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

TARGET = "loan_status"
ID_COL = "person_id"

print("Train shape             :", train_df.shape)
print("Test shape              :", test_df.shape)
print("Sample submission shape :", sample_submission.shape)

display(train_df.head())


In [ ]:
summary_table = pd.DataFrame({
    "dataset": ["train", "test", "sample_submission"],
    "rows": [len(train_df), len(test_df), len(sample_submission)],
    "columns": [train_df.shape[1], test_df.shape[1], sample_submission.shape[1]],
    "duplicate_rows": [
        int(train_df.duplicated().sum()),
        int(test_df.duplicated().sum()),
        int(sample_submission.duplicated().sum()),
    ],
})
summary_table


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Exploratory Data Analysis <a name="5"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

EDA difokuskan pada keputusan yang memengaruhi preprocessing dan modelling: tipe fitur, missing value, keseimbangan kelas, distribusi numerik, serta kategori yang tersedia.


## EDA-1 — Tipe Data dan Missing Value


In [ ]:
dtype_table = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "missing_train": train_df.isna().sum(),
    "missing_test": test_df.reindex(columns=train_df.columns).isna().sum(),
    "n_unique_train": train_df.nunique(),
})
display(dtype_table)

print("Total missing train:", int(train_df.isna().sum().sum()))
print("Total missing test :", int(test_df.isna().sum().sum()))


**Kesimpulan EDA-1**

- Dataset berisi kombinasi fitur numerik dan kategorikal.
- Pipeline preprocessing tetap menyediakan penanganan missing value agar implementasi robust meskipun dataset saat ini tidak memiliki nilai kosong.
- Kolom target hanya tersedia pada train, sedangkan `person_id` tersedia pada train dan test sebagai identifier.


## EDA-2 — Distribusi Target dan Class Imbalance


In [ ]:
target_count = train_df[TARGET].value_counts().sort_index()
target_ratio = train_df[TARGET].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(target_count.index.astype(str), target_count.values)
ax.set_title("Distribusi Target loan_status")
ax.set_xlabel("Kelas")
ax.set_ylabel("Jumlah Baris")
for i, value in enumerate(target_count.values):
    ax.text(i, value, f"{value:,}\n({target_ratio.iloc[i]:.1%})", ha="center", va="bottom")
plt.show()

pd.DataFrame({"count": target_count, "ratio": target_ratio})


**Kesimpulan EDA-2**

Kelas `1` lebih sedikit daripada kelas `0`, sehingga accuracy saja tidak cukup. Evaluasi utama juga mencakup **precision, recall, F1-score, dan ROC-AUC**. Logistic Regression dan SVM menggunakan class weight seimbang yang dihitung secara manual pada implementasi NumPy.


## EDA-3 — Statistik Fitur Numerik


In [ ]:
feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL]]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Numerical features  :", numeric_cols)
print("Categorical features:", categorical_cols)

display(train_df[numeric_cols].describe().T)


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
axes = np.asarray(axes).reshape(-1)

for ax, col in zip(axes, numeric_cols):
    ax.hist(train_df.loc[train_df[TARGET] == 0, col], bins=30, alpha=0.55, density=True, label="0")
    ax.hist(train_df.loc[train_df[TARGET] == 1, col], bins=30, alpha=0.55, density=True, label="1")
    ax.set_title(col)
    ax.legend()

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## EDA-4 — Fitur Kategorikal


In [ ]:
for col in categorical_cols:
    table = pd.crosstab(train_df[col], train_df[TARGET], normalize="index")
    print(f"\nProporsi target per kategori — {col}")
    display(table)


**Keputusan preprocessing dari EDA**

1. Fitur numerik diisi dengan median dan distandardisasi menggunakan statistik training fold.
2. Fitur kategorikal diubah menjadi one-hot encoding menggunakan kategori yang dipelajari dari training fold.
3. Kategori yang tidak pernah muncul di training otomatis menghasilkan seluruh dummy bernilai nol.
4. `person_id` dikeluarkan dari matriks fitur.
5. Split validasi dibuat secara stratified agar rasio kelas tetap konsisten.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Preprocessing & Baseline Validation <a name="6"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Split 80/20 pada bagian ini dipertahankan sebagai **baseline** agar hasil versi lama tetap dapat direproduksi. Pemilihan model final pada versi robust tidak bergantung pada satu holdout ini; keputusan final dilakukan pada bagian Repeated OOF Cross-Validation.


## Stratified Holdout Split From Scratch — Baseline Reproduction


In [ ]:
from dtl_lr_svm import stratified_train_validation_split


## Preprocessor Tabular From Scratch


In [ ]:
from dtl_lr_svm import ScratchTabularPreprocessor


In [ ]:
X_frame = train_df[feature_cols].copy()
y = train_df[TARGET].to_numpy(dtype=int)

preprocessor = ScratchTabularPreprocessor()
X_train = preprocessor.fit_transform(X_frame.iloc[train_idx])
X_valid = preprocessor.transform(X_frame.iloc[valid_idx])
y_train = y[train_idx]
y_valid = y[valid_idx]

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("Jumlah feature setelah encoding:", len(preprocessor.feature_names_))
print(preprocessor.feature_names_)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Implementasi From Scratch <a name="7"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">


## 1. Decision Tree Learning — CART

CART memilih split yang paling besar menurunkan impurity. Untuk klasifikasi biner, Gini impurity adalah:

\[
Gini(S)=1-p_0^2-p_1^2=2p_1(1-p_1)
\]

Gain sebuah split dihitung sebagai impurity parent dikurangi weighted impurity kedua child. Implementasi berikut:

- mencari threshold di antara dua nilai fitur yang berbeda;
- membatasi jumlah kandidat threshold agar efisien;
- menghentikan pertumbuhan berdasarkan `max_depth`, `min_samples_split`, dan `min_samples_leaf`;
- menyimpan probabilitas positif pada leaf.


In [ ]:
from dtl_lr_svm import CARTClassifierScratch


## 2. Logistic Regression

Probabilitas kelas positif diperoleh melalui fungsi sigmoid:

\[
\hat{p}=\sigma(Xw+b)=\frac{1}{1+e^{-(Xw+b)}}
\]

Parameter dioptimalkan dengan weighted binary cross-entropy dan regularisasi L2. Bobot kelas dihitung manual agar kelas minoritas tidak diabaikan. Optimizer Adam juga ditulis langsung menggunakan NumPy.


In [ ]:
from dtl_lr_svm import LogisticRegressionScratch


## 3. Linear Support Vector Machine

Label diubah menjadi \(-1\) dan \(+1\). SVM mencari hyperplane dengan margin besar melalui objective:

\[
\frac{\lambda}{2}\lVert w\rVert^2+
\frac{1}{n}\sum_i \max(0,1-y_i(w^Tx_i+b))
\]

Bagian \(\max(0,1-margin)\) disebut **hinge loss**. Gradien hanya menerima kontribusi dari observasi yang marginnya kurang dari 1.


In [ ]:
from dtl_lr_svm import LinearSVMScratch


## Training Ketiga Model From Scratch


In [ ]:
scratch_model_factories = {
    "CART Scratch": lambda: CARTClassifierScratch(
        max_depth=7,
        min_samples_split=30,
        min_samples_leaf=15,
        max_thresholds=48,
        random_state=SEED,
    ),
    "Logistic Regression Scratch": lambda: LogisticRegressionScratch(
        learning_rate=0.03,
        epochs=250,
        batch_size=512,
        l2=0.001,
        class_weight="balanced",
        random_state=SEED,
    ),
    "Linear SVM Scratch": lambda: LinearSVMScratch(
        learning_rate=0.01,
        epochs=250,
        batch_size=512,
        regularization=0.001,
        class_weight="balanced",
        random_state=SEED,
    ),
}

scratch_models = {}
scratch_train_times = {}

for model_name, factory in scratch_model_factories.items():
    model = factory()
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start_time
    scratch_models[model_name] = model
    scratch_train_times[model_name] = elapsed
    print(f"{model_name:30s} selesai dalam {elapsed:.4f} detik")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(scratch_models["Logistic Regression Scratch"].loss_history_)
axes[0].set_title("Training Loss — Logistic Regression Scratch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Weighted BCE + L2")

axes[1].plot(scratch_models["Linear SVM Scratch"].loss_history_)
axes[1].set_title("Training Loss — Linear SVM Scratch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Hinge Loss + L2")

plt.tight_layout()
plt.show()


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pembanding Scikit-learn <a name="8"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Pembanding menggunakan keluarga algoritma yang sama dan data preprocessing yang identik:

- `DecisionTreeClassifier` untuk CART;
- `LogisticRegression` untuk logistic regression;
- `LinearSVC` untuk linear SVM.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

sklearn_model_factories = {
    "CART Scikit-learn": lambda: DecisionTreeClassifier(
        criterion="gini",
        max_depth=7,
        min_samples_split=30,
        min_samples_leaf=15,
        random_state=SEED,
    ),
    "Logistic Regression Scikit-learn": lambda: LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=SEED,
    ),
    "Linear SVM Scikit-learn": lambda: LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=10000,
        random_state=SEED,
    ),
}

sklearn_models = {}
sklearn_train_times = {}

for model_name, factory in sklearn_model_factories.items():
    model = factory()
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start_time
    sklearn_models[model_name] = model
    sklearn_train_times[model_name] = elapsed
    print(f"{model_name:35s} selesai dalam {elapsed:.4f} detik")


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Evaluasi Baseline <a name="9"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini mereproduksi perbandingan awal pada satu holdout. Hasilnya berguna sebagai baseline, tetapi **bukan satu-satunya dasar pemilihan submission final**.


## Fungsi Evaluasi


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)


def get_model_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def evaluate_model(model_name, implementation, model, X, y, train_time):
    
    prediction = model.predict(X)
    score = get_model_score(model, X)

    return {
        "model": model_name,
        "implementation": implementation,
        "accuracy": accuracy_score(y, prediction),
        "precision_class_1": precision_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        "recall_class_1": recall_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        "f1_class_0": f1_score(
            y, prediction, pos_label=0, zero_division=0
        ),
        "f1_class_1": f1_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        # Metrik resmi kompetisi: rata-rata F1 dari kelas 0 dan kelas 1.
        "macro_f1": f1_score(
            y, prediction, average="macro", zero_division=0
        ),
        "roc_auc": roc_auc_score(y, score),
        "train_time_seconds": train_time,
    }


In [ ]:
evaluation_rows = []

for name, model in scratch_models.items():
    evaluation_rows.append(
        evaluate_model(
            name, "From Scratch", model,
            X_valid, y_valid, scratch_train_times[name]
        )
    )

for name, model in sklearn_models.items():
    evaluation_rows.append(
        evaluate_model(
            name, "Scikit-learn", model,
            X_valid, y_valid, sklearn_train_times[name]
        )
    )

# Ranking utama mengikuti metrik resmi kompetisi: macro F1.
results_df = pd.DataFrame(evaluation_rows).sort_values(
    ["macro_f1", "roc_auc"], ascending=False
).reset_index(drop=True)

results_df.style.format({
    "accuracy": "{:.4f}",
    "precision_class_1": "{:.4f}",
    "recall_class_1": "{:.4f}",
    "f1_class_0": "{:.4f}",
    "f1_class_1": "{:.4f}",
    "macro_f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "train_time_seconds": "{:.4f}",
}).background_gradient(
    subset=["f1_class_0", "f1_class_1", "macro_f1"],
    cmap="YlGn"
)


## Perbandingan Metrik


In [ ]:
metric_columns = [
    "accuracy", "f1_class_0", "f1_class_1", "macro_f1", "roc_auc"
]
plot_data = results_df.set_index("model")[metric_columns]

ax = plot_data.plot(kind="bar", figsize=(15, 6))
ax.set_ylim(0.5, 1.0)
ax.set_title("Perbandingan Model — Macro F1 sebagai Metrik Utama")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Confusion Matrix — Semua Model


In [ ]:
all_models = {**scratch_models, **sklearn_models}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.reshape(-1)

for ax, (name, model) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_valid, model.predict(X_valid))
    image = ax.imshow(cm)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


## ROC Curve


In [ ]:
plt.figure(figsize=(9, 6))
for name, model in all_models.items():
    score = get_model_score(model, X_valid)
    fpr, tpr, _ = roc_curve(y_valid, score)
    auc_value = roc_auc_score(y_valid, score)
    plt.plot(fpr, tpr, label=f"{name} ({auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## Feature Importance CART Scratch


In [ ]:
cart_scratch = scratch_models["CART Scratch"]
importance_df = pd.DataFrame({
    "feature": preprocessor.feature_names_,
    "importance": cart_scratch.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(12))

ax = importance_df.head(12).sort_values("importance").plot(
    kind="barh", x="feature", y="importance", figsize=(9, 6), legend=False
)
ax.set_title("Top Feature Importance — CART Scratch")
ax.set_xlabel("Normalized Gini Gain")
plt.tight_layout()
plt.show()


## Classification Report Model Scratch Terbaik


In [ ]:
scratch_results = results_df[
    results_df["implementation"] == "From Scratch"
].copy()
best_scratch_name = scratch_results.iloc[0]["model"]
best_scratch_model = scratch_models[best_scratch_name]

print("Model scratch terbaik berdasarkan macro F1:", best_scratch_name)
print(
    "Macro F1 validation:",
    f"{scratch_results.iloc[0]['macro_f1']:.5f}"
)
print()
print(classification_report(
    y_valid,
    best_scratch_model.predict(X_valid),
    digits=4,
))


**Interpretasi evaluasi**

- Kompetisi memakai **macro F1-score**, yaitu rata-rata F1 kelas `0` dan F1 kelas `1`. Karena kedua kelas diberi bobot yang sama, performa pada kelas mayoritas tidak dapat menutupi performa yang buruk pada kelas minoritas.
- Model scratch terbaik dan file submission utama dipilih berdasarkan `macro_f1`, bukan accuracy maupun F1 kelas `1` saja.
- `f1_class_0` dan `f1_class_1` tetap ditampilkan agar keseimbangan performa antar-kelas dapat diperiksa.
- ROC-AUC menilai kualitas ranking score di seluruh threshold, sedangkan confusion matrix menunjukkan false positive dan false negative secara langsung.
- Perbedaan hasil scratch dan scikit-learn dapat muncul karena optimisasi, strategi threshold, stopping criterion, regularisasi, dan detail implementasi numerik.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Robust Model Development V2 — Weighted CART, No Leakage <a name="10"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Target kompetisi adalah **macro F1**, sementara kelas positif lebih sedikit. Pada V1, CART memakai impurity biasa lalu hanya menyesuaikan decision threshold. V2 menambahkan **cost-sensitive CART**: bobot kelas positif ikut masuk ke perhitungan Gini ketika tree memilih split.

Perbaikan utama:

1. **Class-weighted Gini pada CART scratch** — positive class weight dituning dari train OOF saja.
2. **Unbounded depth + regularisasi minimum leaf** — tree boleh tumbuh selama split masih valid, tetapi `min_samples_leaf` mencegah leaf terlalu kecil.
3. **Exact split search** — seluruh threshold numerik yang valid diperiksa.
4. **5-fold OOF tuning + repeated-seed stability audit**.
5. **Exact macro-F1 threshold optimization** pada prediksi OOF.
6. Logistic Regression dan Linear SVM tetap dipertahankan sebagai implementasi scratch dan pembanding yang diwajibkan.

> Class weighting bukan ensemble dan bukan algoritma tambahan. Submission tetap berasal dari **satu CART from scratch**. `person_id` tetap tidak digunakan sebagai feature.


## Utilitas Stratified K-Fold dan Exact Threshold Optimization

Macro F1 sensitif terhadap threshold. Threshold `0.5` pada Logistic Regression atau `0.0` pada SVM tidak selalu optimal, terutama saat kelas tidak seimbang. Fungsi berikut mencari threshold terbaik dari OOF score secara `O(n log n)` melalui sorting dan cumulative confusion counts.


In [ ]:
from dtl_lr_svm import (
    macro_f1_numpy,
    optimize_macro_f1_threshold,
    stratified_kfold_indices,
)


## Nonlinear Feature Expansion untuk Logistic Regression dan SVM

CART sudah bersifat nonlinear sehingga menggunakan fitur dasar. Untuk model linear, ditambahkan basis threshold berbasis quantile dan interaksi dengan `person_home_ownership`. Transformasi ini **bukan model baru**: prediktor akhirnya tetap Logistic Regression atau Linear SVM. Cut-point quantile dipelajari tanpa melihat `y` dan selalu di-fit ulang pada training fold.


In [ ]:
from dtl_lr_svm import ScratchNonlinearPreprocessor


## Hyperparameter Search V2 — Fokus pada Cost-Sensitive CART

Grid V2 menambahkan `positive_class_weight` pada CART. Bobot ini hanya mengubah kontribusi kelas positif di **Gini impurity** dan estimasi probabilitas leaf; bukan resampling, boosting, bagging, atau ensemble.

Semua kandidat dipilih dari **train OOF**. `test.csv` dan public leaderboard tidak dipakai untuk memilih parameter atau threshold.


In [ ]:
ROBUST_TUNING_GRIDS = {
    "CART Scratch": [
        {"max_depth": 8, "min_samples_leaf": 40, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 30, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.1},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.2},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.3},
        {"max_depth": None, "min_samples_leaf": 50, "positive_class_weight": 1.3},
    ],
    "Logistic Regression Scratch": [
        {"l2": 0.001},
        {"l2": 0.003},
    ],
    "Linear SVM Scratch": [
        {"regularization": 0.001},
        {"regularization": 0.003},
    ],
}


def make_robust_preprocessor(model_name):
    if model_name == "CART Scratch":
        return ScratchTabularPreprocessor()
    if model_name == "Logistic Regression Scratch":
        return ScratchNonlinearPreprocessor(n_quantiles=15)
    return ScratchNonlinearPreprocessor(n_quantiles=9)


def make_robust_model(model_name, params, seed=SEED):
    if model_name == "CART Scratch":
        leaf = int(params["min_samples_leaf"])
        depth = params.get("max_depth", None)
        return CARTClassifierScratch(
            max_depth=None if depth is None else int(depth),
            min_samples_split=2 * leaf,
            min_samples_leaf=leaf,
            max_thresholds=None,
            positive_class_weight=float(params.get("positive_class_weight", 1.0)),
            random_state=seed,
        )

    if model_name == "Logistic Regression Scratch":
        return LogisticRegressionScratch(
            learning_rate=0.02,
            epochs=180,
            batch_size=512,
            l2=float(params["l2"]),
            class_weight=None,
            patience=25,
            random_state=seed,
        )

    return LinearSVMScratch(
        learning_rate=0.01,
        epochs=60,
        batch_size=512,
        regularization=float(params["regularization"]),
        class_weight=None,
        random_state=seed,
    )


def run_oof_cv(model_name, params, cv_seed=2026, n_splits=5):
    y_all = train_df[TARGET].to_numpy(dtype=int)
    oof_score = np.zeros(len(train_df), dtype=float)
    fold_rows = []
    start_time = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(
        stratified_kfold_indices(y_all, n_splits=n_splits, random_state=cv_seed),
        start=1,
    ):
        pre = make_robust_preprocessor(model_name)
        X_tr = pre.fit_transform(train_df.iloc[tr_idx][feature_cols])
        X_va = pre.transform(train_df.iloc[va_idx][feature_cols])

        model = make_robust_model(model_name, params, seed=SEED + fold_id)
        model.fit(X_tr, y_all[tr_idx])
        score = get_model_score(model, X_va)
        oof_score[va_idx] = score

        default_threshold = 0.0 if model_name == "Linear SVM Scratch" else 0.5
        fold_rows.append({
            "fold": fold_id,
            "macro_f1_default": macro_f1_numpy(
                y_all[va_idx], (score >= default_threshold).astype(int)
            ),
        })

    threshold, macro_f1_oof = optimize_macro_f1_threshold(y_all, oof_score)
    elapsed = time.perf_counter() - start_time

    return {
        "model": model_name,
        "params": params.copy(),
        "cv_seed": cv_seed,
        "threshold": threshold,
        "macro_f1_oof": macro_f1_oof,
        "roc_auc_oof": roc_auc_score(y_all, oof_score),
        "time_seconds": elapsed,
        "oof_score": oof_score,
        "fold_rows": fold_rows,
    }


In [ ]:
# Phase A — compact 5-fold tuning pada satu seed.
TUNING_SEED = 2026
tuning_results = []
robust_cv_cache = {}

for model_name, grid in ROBUST_TUNING_GRIDS.items():
    for params in grid:
        result = run_oof_cv(
            model_name=model_name,
            params=params,
            cv_seed=TUNING_SEED,
            n_splits=5,
        )
        tuning_results.append({
            "model": model_name,
            "params": str(params),
            "threshold": result["threshold"],
            "macro_f1_oof": result["macro_f1_oof"],
            "roc_auc_oof": result["roc_auc_oof"],
            "time_seconds": result["time_seconds"],
        })
        robust_cv_cache[(model_name, str(params), TUNING_SEED)] = result
        print(
            f"{model_name:30s} {params} -> "
            f"OOF macro F1={result['macro_f1_oof']:.5f} | "
            f"threshold={result['threshold']:.5f}"
        )

robust_tuning_df = pd.DataFrame(tuning_results).sort_values(
    ["model", "macro_f1_oof"], ascending=[True, False]
).reset_index(drop=True)

display(robust_tuning_df)


## Stability Audit — Repeated 5-Fold CV pada Dua Kandidat Teratas

Dua konfigurasi dengan OOF macro F1 tertinggi dari Phase A diuji ulang pada tiga seed 5-fold. Kandidat final dipilih berdasarkan **mean repeated OOF macro F1**, lalu threshold final diambil dari median threshold antar-repeat. Repeated CV hanya dipakai untuk evaluasi/tuning; submission tetap berasal dari satu model manual.


In [ ]:
best_params_by_model = {}
best_threshold_by_model = {}
best_oof_by_model = {}

for model_name in ROBUST_TUNING_GRIDS:
    family_rows = robust_tuning_df[
        robust_tuning_df["model"] == model_name
    ].sort_values("macro_f1_oof", ascending=False)
    best_row = family_rows.iloc[0]
    best_params_text = best_row["params"]
    best_params_by_model[model_name] = next(
        p for p in ROBUST_TUNING_GRIDS[model_name]
        if str(p) == best_params_text
    )
    best_threshold_by_model[model_name] = float(best_row["threshold"])
    best_oof_by_model[model_name] = float(best_row["macro_f1_oof"])

family_best_df = pd.DataFrame([
    {
        "model": model_name,
        "params": str(best_params_by_model[model_name]),
        "macro_f1_oof": best_oof_by_model[model_name],
        "threshold": best_threshold_by_model[model_name],
    }
    for model_name in ROBUST_TUNING_GRIDS
]).sort_values("macro_f1_oof", ascending=False).reset_index(drop=True)

display(family_best_df)

# Tiga konfigurasi terbaik secara global masuk stability audit.
finalist_df = robust_tuning_df.sort_values(
    ["macro_f1_oof", "roc_auc_oof"], ascending=False
).head(3).reset_index(drop=True)

STABILITY_SEEDS = [42, 2026, 777]
stability_rows = []

for finalist_id, row in finalist_df.iterrows():
    model_name = row["model"]
    params_text = row["params"]
    params = next(
        p for p in ROBUST_TUNING_GRIDS[model_name]
        if str(p) == params_text
    )

    for cv_seed in STABILITY_SEEDS:
        cache_key = (model_name, str(params), cv_seed)
        result = robust_cv_cache.get(cache_key)
        if result is None:
            result = run_oof_cv(
                model_name=model_name,
                params=params,
                cv_seed=cv_seed,
                n_splits=5,
            )
        stability_rows.append({
            "candidate_id": finalist_id,
            "model": model_name,
            "params": str(params),
            "cv_seed": cv_seed,
            "threshold": result["threshold"],
            "macro_f1_oof": result["macro_f1_oof"],
            "roc_auc_oof": result["roc_auc_oof"],
        })

stability_df = pd.DataFrame(stability_rows)
display(stability_df.sort_values(["candidate_id", "cv_seed"]))

stability_summary = (
    stability_df.groupby(["candidate_id", "model", "params"], as_index=False)
    .agg(
        mean_macro_f1=("macro_f1_oof", "mean"),
        std_macro_f1=("macro_f1_oof", "std"),
        mean_roc_auc=("roc_auc_oof", "mean"),
        median_threshold=("threshold", "median"),
    )
    .sort_values(["mean_macro_f1", "mean_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

display(stability_summary.style.format({
    "mean_macro_f1": "{:.5f}",
    "std_macro_f1": "{:.5f}",
    "mean_roc_auc": "{:.5f}",
    "median_threshold": "{:.5f}",
}))

best_stability_row = stability_summary.iloc[0]
best_robust_model_name = best_stability_row["model"]
best_params_text = best_stability_row["params"]
best_robust_params = next(
    p for p in ROBUST_TUNING_GRIDS[best_robust_model_name]
    if str(p) == best_params_text
)
best_robust_threshold = float(best_stability_row["median_threshold"])

# Threshold model non-final memakai hasil best 5-fold Phase A; kandidat final
 # memakai median repeated-CV agar lebih stabil.
threshold_by_model = best_threshold_by_model.copy()
threshold_by_model[best_robust_model_name] = best_robust_threshold
best_params_by_model[best_robust_model_name] = best_robust_params

print("Model robust terpilih  :", best_robust_model_name)
print("Hyperparameter         :", best_robust_params)
print("Median OOF threshold   :", f"{best_robust_threshold:.6f}")
print("Mean repeated OOF F1   :", f"{best_stability_row['mean_macro_f1']:.6f}")
print("Std repeated OOF F1    :", f"{best_stability_row['std_macro_f1']:.6f}")


### Mengapa V2 Lebih Robust?

- `person_id` **tetap tidak digunakan** sebagai feature walaupun identifier dapat mengandung pola artifisial pada dataset.
- Class weighting dipelajari sebagai hyperparameter dari train OOF dan diterapkan langsung dalam Gini impurity CART.
- `max_depth=None` tidak berarti tree tanpa regularisasi: `min_samples_leaf=40` dan `min_samples_split=80` tetap membatasi kompleksitas.
- Threshold klasifikasi dipilih dari OOF prediction, bukan dari test/public leaderboard.
- Preprocessing selalu di-fit ulang pada training fold.
- Model final tetap **satu CART scratch**; repeated CV hanya untuk model selection, bukan averaging prediction.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Final Training & Kaggle Submission <a name="11"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Setelah hyperparameter dan threshold dibekukan dari repeated OOF CV, setiap keluarga model scratch dilatih **satu kali** pada seluruh `train.csv`. Submission utama berasal dari model dengan mean repeated OOF macro F1 tertinggi. Tidak ada ensemble pada tahap submission.


## Fit Final Model pada Seluruh Train — Konfigurasi Sudah Dibekukan


In [ ]:
y_full = train_df[TARGET].to_numpy(dtype=int)

final_family_artifacts = {}
for model_name in ROBUST_TUNING_GRIDS:
    params = best_params_by_model[model_name]
    threshold = float(threshold_by_model[model_name])

    preprocessor_family = make_robust_preprocessor(model_name)
    X_full_family = preprocessor_family.fit_transform(train_df[feature_cols])
    X_test_family = preprocessor_family.transform(test_df[feature_cols])

    model_family = make_robust_model(model_name, params, seed=SEED)
    model_family.fit(X_full_family, y_full)
    test_score_family = get_model_score(model_family, X_test_family)
    test_prediction_family = (test_score_family >= threshold).astype(int)

    final_family_artifacts[model_name] = {
        "preprocessor": preprocessor_family,
        "model": model_family,
        "params": params,
        "threshold": threshold,
        "test_score": test_score_family,
        "test_prediction": test_prediction_family,
    }

    print(
        f"{model_name:30s} | features={X_full_family.shape[1]:4d} | "
        f"threshold={threshold:.5f} | positive={test_prediction_family.mean():.2%}"
    )


## Buat Submission untuk Ketiga Model Scratch Robust


In [ ]:
final_submission_paths = {}
final_models = {}

for model_name, artifact in final_family_artifacts.items():
    test_prediction = artifact["test_prediction"].astype(int)
    short_name = (
        model_name.lower()
        .replace(" scratch", "")
        .replace(" ", "_")
    )
    submission_path = OUTPUT_DIR / f"submission_{short_name}_robust_scratch.csv"

    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL].to_numpy(),
        TARGET: test_prediction,
    })
    submission.to_csv(submission_path, index=False)

    final_models[model_name] = artifact["model"]
    final_submission_paths[model_name] = submission_path
    print(f"{model_name:30s} -> {submission_path.name}")


## Pilih Submission Utama Berdasarkan Repeated OOF Macro F1


In [ ]:
best_submission_source = final_submission_paths[best_robust_model_name]
best_submission = pd.read_csv(best_submission_source)
best_submission_path = OUTPUT_DIR / "submission_best_scratch_weighted_cart_v2.csv"
best_submission.to_csv(best_submission_path, index=False)

assert list(best_submission.columns) == list(sample_submission.columns), (
    "Kolom submission tidak sesuai sample_submission."
)
assert len(best_submission) == len(test_df), "Jumlah baris submission tidak sesuai test."
assert np.array_equal(
    best_submission[ID_COL].to_numpy(),
    test_df[ID_COL].to_numpy(),
), "Urutan person_id berubah."
assert set(best_submission[TARGET].unique()).issubset({0, 1}), (
    "Prediksi harus berupa label 0 atau 1."
)

print("Model submission utama :", best_robust_model_name)
print("Hyperparameter          :", best_robust_params)
print("OOF threshold           :", f"{best_robust_threshold:.6f}")
print("File submission utama  :", best_submission_path)
print("Shape                   :", best_submission.shape)
display(best_submission.head(10))


## Melakukan Submission Minimal Satu Kali

File utama V2 yang dibuat notebook:

```text
/kaggle/working/submission_best_scratch_weighted_cart_v2.csv
```

Submission ini berasal dari **satu model CART from scratch**, bukan ensemble. Submit file tersebut melalui panel Output Kaggle atau aktifkan cell API berikut setelah kredensial tersedia.


In [ ]:
# Ubah menjadi True hanya ketika Kaggle API sudah terautentikasi.
SUBMIT_TO_KAGGLE = False
COMPETITION_NAME = "ai-lab-recruitment-task-2"
SUBMISSION_MESSAGE = (
    f"Robust V2 weighted-CART from-scratch {best_robust_model_name}; "
    f"repeated OOF threshold={best_robust_threshold:.4f}"
)

if SUBMIT_TO_KAGGLE:
    import subprocess

    command = [
        "kaggle", "competitions", "submit",
        "-c", COMPETITION_NAME,
        "-f", str(best_submission_path),
        "-m", SUBMISSION_MESSAGE,
    ]
    completed = subprocess.run(command, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
else:
    print(
        "Submission API belum dijalankan. Set SUBMIT_TO_KAGGLE=True "
        "setelah Kaggle API terautentikasi, atau submit file melalui panel Output."
    )


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Kesimpulan <a name="12"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

V2 meningkatkan notebook tanpa keluar dari spesifikasi tugas:

1. **CART, Logistic Regression, dan Linear SVM tetap dibuat from scratch** dengan NumPy.
2. CART V2 memakai **class-weighted Gini**, exact split search, `min_samples_leaf`, dan OOF threshold tuning.
3. Class weighting merupakan modifikasi loss/impurity pada Decision Tree, **bukan ensemble method** dan bukan algoritma tambahan.
4. Model final dipilih dari repeated OOF CV dan dilatih satu kali pada seluruh train.
5. `person_id` tidak digunakan sebagai predictor.
6. Test/public leaderboard tidak digunakan untuk tuning parameter.
7. Target praktis V2 adalah memperbaiki macro F1 terutama pada kelas minoritas tanpa mengorbankan stabilitas kelas mayoritas.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Optional Post-Submission Audit <a name="13"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian evaluasi eksternal tidak menjadi bagian dari notebook ini. Seluruh tuning, feature engineering, threshold selection, dan pemilihan model hanya memakai data train.


In [ ]:
# ================================================================
# OPTIONAL POST-SUBMISSION AUDIT — DISABLED BY DEFAULT
# ================================================================
EVALUATE_EXTERNAL_LABELS = False

EXTERNAL_LABELS_KAGGLE_PATH = Path(
    "/kaggle/input/datasets/kurtmikhael/external-labels/test_labels.csv"
)
EXTERNAL_LABELS_LOCAL_PATH = Path("/mnt/data/test_labels.csv")
EXTERNAL_LABELS_PATH = EXTERNAL_LABELS_KAGGLE_PATH if EXTERNAL_LABELS_KAGGLE_PATH.exists() else EXTERNAL_LABELS_LOCAL_PATH

if not EVALUATE_EXTERNAL_LABELS:
    print(
        "Evaluasi eksternal dinonaktifkan untuk menjaga proses modelling no-leakage. "
        "Aktifkan hanya SETELAH model/hyperparameter/threshold final dibekukan dan submission dilakukan."
    )
else:
    if not EXTERNAL_LABELS_PATH.exists():
        raise FileNotFoundError("Label eksternal tidak ditemukan untuk evaluasi.")

    external_labels_df = pd.read_csv(EXTERNAL_LABELS_PATH)[[ID_COL, TARGET]].copy()
    audit_df = best_submission.merge(
        external_labels_df,
        on=ID_COL,
        how="inner",
        suffixes=("_prediction", "_truth"),
        validate="one_to_one",
    )

    y_truth = audit_df[f"{TARGET}_truth"].to_numpy(dtype=int)
    y_pred = audit_df[f"{TARGET}_prediction"].to_numpy(dtype=int)

    print("POST-SUBMISSION AUDIT ONLY")
    print("Rows matched :", len(audit_df))
    print("Macro F1     :", f"{f1_score(y_truth, y_pred, average='macro'):.6f}")
    print("Accuracy     :", f"{accuracy_score(y_truth, y_pred):.6f}")
    print()
    print(classification_report(y_truth, y_pred, digits=4))
